# GENESIS 369 — Exclusive Model Training (Colab Free T4)

**What this builds:** the exclusive 369 language model — GENESIS's own mind, trained on YZ's real dialogues.

**Pipeline (research-verified, zero spend):**
1. Base: `Qwen/Qwen3-8B` (Apache-2.0, open license)
2. **Stage 1 — QLoRA SFT** (r=64, alpha=128): learns 369 knowledge + GENESIS tone (445 dialogue pairs)
3. **Stage 2 — QLoRA DPO** (r=32, alpha=64, beta=0.1): identity lock — learns what it must never break (38 preference pairs)
4. Adapters + tokenizer saved to Google Drive (`/MyDrive/genesis_369_model/`)

Run all cells top to bottom. Total runtime on a free T4: well under one session.

In [ ]:
# 0 — GPU check (fail fast with a clear message if Colab gave us no GPU)
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True)
gpu = out.stdout.strip()
print("GPU:", gpu if gpu else "NONE DETECTED")
assert gpu, ("NO GPU — Colab assigned a CPU runtime. Fix: Runtime > Change runtime type > "
             "Hardware accelerator = T4 GPU > Save, then Runtime > Run all again.")
print("OK — GPU present, training can proceed.")

In [ ]:
# 1 — Install training stack (Unsloth: fastest QLoRA on T4, purpose-built for Colab)
%%capture
import torch
major_version, minor_version = torch.cuda.get_device_capability()
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
print("Install done — if this is the first run, Colab may ask to restart the runtime; "
      "if so, restart and run all cells again (installs persist for the session).")

In [ ]:
# 2 — Mount Google Drive (checkpoints land here, so nothing is lost if the session ends)
from google.colab import drive
drive.mount('/content/drive')
import os
OUT_DIR = "/content/drive/MyDrive/genesis_369_model"
os.makedirs(OUT_DIR, exist_ok=True)
print("Checkpoints will be saved to:", OUT_DIR)

In [ ]:
# 3 — Download the 369 dataset (committed, verified: 445 SFT + 38 DPO)
SFT_URL = "https://raw.githubusercontent.com/369369yz-collab/genesis-windows/main/model/dataset/369_sft_full.jsonl"
DPO_URL = "https://raw.githubusercontent.com/369369yz-collab/genesis-windows/main/model/dataset/369_dpo_full.jsonl"
import urllib.request, json
def fetch_jsonl(url, path):
    urllib.request.urlretrieve(url, path)
    rows = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
    print(f"{path}: {len(rows)} records")
    return rows
sft_rows = fetch_jsonl(SFT_URL, "/content/369_sft_full.jsonl")
dpo_rows = fetch_jsonl(DPO_URL, "/content/369_dpo_full.jsonl")
assert len(sft_rows) == 445, f"SFT count drift: {len(sft_rows)} != 445"
assert len(dpo_rows) == 38, f"DPO count drift: {len(dpo_rows)} != 38"
print("Dataset verified: 445 SFT + 38 DPO — no drift.")

In [ ]:
# 4 — Load base model (4-bit QLoRA-ready) + tokenizer
from unsloth import FastLanguageModel
BASE_MODEL = "Qwen/Qwen3-8B"
MAX_SEQ = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ,
    dtype=None,          # auto: bfloat16 on T4
    load_in_4bit=True,
)
print("Base loaded:", BASE_MODEL)
print("Chat template:", tokenizer.chat_template is not None)

In [ ]:
# 5 — Prepare SFT data: apply chat template to each dialogue -> 'text'
EOS = tokenizer.eos_token
def to_text(messages):
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False) + EOS
sft_texts = [to_text(r["messages"]) for r in sft_rows]
from datasets import Dataset
sft_ds = Dataset.from_dict({"text": sft_texts})
print("SFT samples:", len(sft_ds))
print("Sample 0 preview:", sft_texts[0][:300].replace(chr(10), " "), "...")

In [ ]:
# 6 — STAGE 1: QLoRA SFT (r=64, alpha=128, all linear targets) — learns 369 knowledge + GENESIS tone
from unsloth import is_bfloat16_supported
model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=128,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=369,
    use_rslora=False,
    loftq_config=None,
)
from trl import SFTTrainer
from transformers import TrainingArguments
sft_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=sft_ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        warmup_steps=10,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=369,
        output_dir="/content/sft_out",
        report_to="none",
    ),
)
sft_trainer.train()
print("STAGE 1 (SFT) complete.")

In [ ]:
# 7 — Save SFT adapter checkpoint to Drive
import os
sft_dir = os.path.join(OUT_DIR, "stage1_sft_adapter")
model.save_pretrained(sft_dir)
tokenizer.save_pretrained(sft_dir)
print("SFT adapter saved to:", sft_dir)

In [ ]:
# 8 — Prepare DPO data: prompt (templated) + chosen + rejected
def dpo_format(row):
    prompt = tokenizer.apply_chat_template(row["prompt"], tokenize=False, add_generation_prompt=True)
    return {"prompt": prompt, "chosen": row["chosen"] + EOS, "rejected": row["rejected"] + EOS}
dpo_dicts = [dpo_format(r) for r in dpo_rows]
dpo_ds = Dataset.from_list(dpo_dicts)
print("DPO pairs:", len(dpo_ds))
print("Prompt preview:", dpo_dicts[0]["prompt"][:200].replace(chr(10), " "), "...")

In [ ]:
# 9 — STAGE 2: QLoRA DPO (r=32, alpha=64, beta=0.1) — identity lock
# Fresh LoRA adapter for the DPO stage, stacked on the SFT-tuned model
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=64,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=369,
    use_rslora=False,
    loftq_config=None,
)
from unsloth import PatchDPOTrainer
PatchDPOTrainer()
from trl import DPOTrainer
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,          # unsloth handles the reference internally
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        warmup_steps=5,
        learning_rate=5e-5,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=369,
        output_dir="/content/dpo_out",
        report_to="none",
    ),
    beta=0.1,
    train_dataset=dpo_ds,
    tokenizer=tokenizer,
    max_length=MAX_SEQ,
    max_prompt_length=MAX_SEQ // 2,
)
dpo_trainer.train()
print("STAGE 2 (DPO) complete — identity locked.")

In [ ]:
# 10 — Save final 369 adapter + tokenizer to Drive, and print the summary
final_dir = os.path.join(OUT_DIR, "genesis_369_final_adapter")
model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)
print("=" * 60)
print("GENESIS 369 exclusive model training COMPLETE")
print("=" * 60)
print("Base:            ", BASE_MODEL)
print("Stage 1 (SFT):   r=64/alpha=128 on 445 dialogue pairs")
print("Stage 2 (DPO):   r=32/alpha=64/beta=0.1 on 38 preference pairs")
print("Final adapter:   ", final_dir)
print()
print("Next: merge adapter into base -> export GGUF Q5_K_M -> load into the")
print("Windows GENESIS llama-server sidecar (final build step).")